# ARC-v0.23.1 — NaN-Safe Statistical Recompute

**Purpose.** Repair the analysis-only NaN handling in ARC-v0.23 without rerunning retrieval.

ARC-v0.23 completed successfully on **3,316 FEVER FIT-held-out validation queries**, producing **1,750,848 query-policy-mechanism-round rows**. The Eq. 6 vector identity and additive utility identity passed to numerical precision. However, some `state_component_cosine` and `state_cancellation_ratio` values are mathematically undefined when one or both component vectors have zero norm. The original query-level point estimates used pandas `mean()` (which skips NaN), but the bootstrap helper converted the aggregated column to NumPy and called `np.mean()` without first dropping undefined query-level values. This caused NaN confidence intervals for selected nprobe summaries.

This notebook:

1. reads the already-frozen v0.23 merged replay parquet;
2. **does not rerun ANN retrieval**;
3. recomputes query-level and round-level summaries;
4. explicitly reports `n_defined`, `n_total`, and missing/undefined fractions;
5. bootstraps only mathematically defined query-level values;
6. recomputes pairwise query-level differences on the paired defined subset;
7. writes an analysis addendum and artifact hashes.

### Interpretation rule

Undefined values are **not imputed** and are **not converted to zero**.  
All cosine/alignment and cancellation-ratio estimates are reported **conditional on being defined**, with coverage shown next to the estimate.

This is a statistical repair of post-primary mechanism hardening, not a new experiment and not a modification of the original frozen retrieval protocol.

In [ ]:
# Cell 1 — Imports / constants
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, math

import numpy as np
import pandas as pd
from google.colab import drive

SEED = 20260823
BOOTSTRAP_REPS = 10_000

MECHANISMS = ["e5_representation", "e5_nprobe", "e5_hnsw"]

MEASURES = [
    "state_prop_norm",
    "state_direct_norm",
    "state_total_norm",
    "state_component_cosine",
    "state_cancellation_ratio",
    "state_prop_cosdist",
    "state_direct_cosdist",
    "candidate_prop_jaccard",
    "candidate_direct_jaccard",
    "feedback_prop_cosdist",
    "feedback_direct_cosdist",
    "utility_prop_abs",
    "utility_direct_abs",
    "utility_prop_signed",
    "utility_direct_signed",
]

CORE_MEASURES = [
    "state_direct_norm",
    "state_prop_norm",
    "state_component_cosine",
    "state_cancellation_ratio",
    "candidate_direct_jaccard",
    "feedback_direct_cosdist",
    "utility_direct_abs",
]

def sha256_file(path, chunk=16*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print("Ready.")

In [ ]:
# Cell 2 — Mount Drive and resolve the completed v0.23 run
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
V023_ROOT = ARC_ROOT / "eq6-common-state-operator-replay-v023"

# Prefer the completed run already used for v0.23:
PREFERRED_RUN = V023_ROOT / "20260822-162028"

def is_complete_v023(p):
    return (
        p.is_dir()
        and (p / "v023_full_common_state_replay.parquet").is_file()
        and (p / "v023_full_final_report.json").is_file()
        and (p / "v023_eq6_common_state_protocol.json").is_file()
    )

if is_complete_v023(PREFERRED_RUN):
    V023_RUN = PREFERRED_RUN
else:
    candidates = sorted([p for p in V023_ROOT.iterdir() if is_complete_v023(p)], reverse=True)
    if not candidates:
        raise FileNotFoundError("No complete ARC-v0.23 run found.")
    V023_RUN = candidates[0]

REPLAY_PATH = V023_RUN / "v023_full_common_state_replay.parquet"
V023_REPORT = V023_RUN / "v023_full_final_report.json"
V023_PROTOCOL = V023_RUN / "v023_eq6_common_state_protocol.json"

print("V023_RUN:", V023_RUN)
print("Replay:", REPLAY_PATH)
print("Replay size MiB:", round(REPLAY_PATH.stat().st_size / 2**20, 2))
print("v0.23 report SHA256:", sha256_file(V023_REPORT))
print("v0.23 protocol SHA256:", sha256_file(V023_PROTOCOL))

In [ ]:
# Cell 3 — Freeze v0.23.1 analysis repair protocol BEFORE repaired summaries
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT_ROOT = ARC_ROOT / "eq6-common-state-operator-replay-v0231"
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

PROTOCOL = {
    "status": "ARC_V0231_NAN_SAFE_RECOMPUTE_PROTOCOL_FROZEN",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "parent_v023_run": str(V023_RUN),
    "parent_replay_sha256": sha256_file(REPLAY_PATH),
    "parent_report_sha256": sha256_file(V023_REPORT),
    "parent_protocol_sha256": sha256_file(V023_PROTOCOL),
    "retrieval_rerun": False,
    "retrieval_parameters_changed": False,
    "contrast_changed": False,
    "policy_grid_changed": False,
    "threshold_changed": False,
    "repair": (
        "Drop mathematically undefined query-level values only for the measure being "
        "estimated; never impute zero. Report n_defined/n_total and undefined fraction."
    ),
    "statistical_unit": "query",
    "bootstrap_reps": BOOTSTRAP_REPS,
    "seed": SEED + 231,
    "pairwise_rule": (
        "For A-B contrasts, use only queries where both A and B query-level summaries "
        "are defined for that measure; report paired n."
    ),
    "round_rule": (
        "Round-specific component cosine may be undefined at round 0 because propagated "
        "state difference is structurally zero. Report coverage rather than forcing a value."
    ),
    "test_accessed": False,
    "test_relevance_accessed": False,
}

PROTOCOL_PATH = OUT / "v0231_nan_safe_protocol.json"
PROTOCOL_PATH.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True))
PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)

print("OUTPUT:", OUT)
print("PROTOCOL SHA:", PROTOCOL_SHA)
print("V0.23.1 PROTOCOL FROZEN — PASS")

In [ ]:
# Cell 4 — Load v0.23 replay and integrity-check lineage
replay = pd.read_parquet(REPLAY_PATH)

expected_rows = 3316 * 44 * 3 * 4
assert len(replay) == expected_rows, (len(replay), expected_rows)
assert replay["query_id"].nunique() == 3316
assert set(replay["mechanism"].unique()) == set(MECHANISMS)

assert replay["eq6_vector_identity_error"].max() < 1e-5
assert replay["utility_additive_identity_error"].max() < 1e-10

print("rows:", f"{len(replay):,}")
print("queries:", replay["query_id"].nunique())
print("max Eq.6 vector identity error:", replay["eq6_vector_identity_error"].max())
print("max utility additive identity error:", replay["utility_additive_identity_error"].max())
print("PARENT REPLAY INTEGRITY — PASS")

In [ ]:
# Cell 5 — Diagnose undefined values at event level
diag_rows = []

for mechanism in MECHANISMS:
    for measure in ["state_component_cosine", "state_cancellation_ratio"]:
        sub = replay.loc[replay["mechanism"].eq(mechanism), ["round", measure]]
        for round_id in sorted(sub["round"].unique()):
            x = sub.loc[sub["round"].eq(round_id), measure]
            diag_rows.append({
                "mechanism": mechanism,
                "round": int(round_id),
                "measure": measure,
                "n_events": int(len(x)),
                "n_defined": int(x.notna().sum()),
                "n_undefined": int(x.isna().sum()),
                "undefined_fraction": float(x.isna().mean()),
            })

event_undefined = pd.DataFrame(diag_rows)
EVENT_UNDEF_PATH = OUT / "v0231_event_level_undefined_diagnostics.csv"
event_undefined.to_csv(EVENT_UNDEF_PATH, index=False)

display(event_undefined)

In [ ]:
# Cell 6 — Query-level aggregation with explicit coverage
# pandas groupby mean skips undefined event-level values within a query.
query_mech = (
    replay.groupby(["query_id", "mechanism"], as_index=False)[MEASURES]
    .mean()
)

query_mech_round = (
    replay.groupby(["query_id", "mechanism", "round"], as_index=False)[MEASURES]
    .mean()
)

coverage_rows = []
for mechanism in MECHANISMS:
    sub = query_mech[query_mech["mechanism"].eq(mechanism)]
    for measure in MEASURES:
        x = sub[measure]
        coverage_rows.append({
            "mechanism": mechanism,
            "measure": measure,
            "n_total_queries": int(len(x)),
            "n_defined_queries": int(x.notna().sum()),
            "n_undefined_queries": int(x.isna().sum()),
            "undefined_fraction": float(x.isna().mean()),
        })

query_coverage = pd.DataFrame(coverage_rows)
QUERY_COVERAGE_PATH = OUT / "v0231_query_level_coverage.csv"
query_coverage.to_csv(QUERY_COVERAGE_PATH, index=False)

display(query_coverage[
    query_coverage["measure"].isin(["state_component_cosine", "state_cancellation_ratio"])
])

In [ ]:
# Cell 7 — NaN-safe query-bootstrap function
rng = np.random.default_rng(SEED + 231)

def bootstrap_defined_mean_ci(series, reps=BOOTSTRAP_REPS):
    s = pd.Series(series)
    x = s.dropna().to_numpy(np.float64)

    n_total = int(len(s))
    n_defined = int(len(x))
    n_undefined = n_total - n_defined

    if n_defined == 0:
        return {
            "mean": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "n_total_queries": n_total,
            "n_defined_queries": 0,
            "n_undefined_queries": n_undefined,
            "undefined_fraction": 1.0 if n_total else np.nan,
        }

    point = float(np.mean(x))
    if n_defined == 1:
        lo = hi = point
    else:
        boots = np.empty(reps, dtype=np.float64)
        for b in range(reps):
            idx = rng.integers(0, n_defined, size=n_defined)
            boots[b] = float(np.mean(x[idx]))
        lo, hi = np.quantile(boots, [0.025, 0.975])

    return {
        "mean": point,
        "ci95_low": float(lo),
        "ci95_high": float(hi),
        "n_total_queries": n_total,
        "n_defined_queries": n_defined,
        "n_undefined_queries": n_undefined,
        "undefined_fraction": n_undefined / n_total if n_total else np.nan,
    }

print("NaN-safe bootstrap helper ready.")

In [ ]:
# Cell 8 — Recompute mechanism-level summaries
summary_rows = []

for mechanism in MECHANISMS:
    sub = query_mech[query_mech["mechanism"].eq(mechanism)]
    for measure in MEASURES:
        stats = bootstrap_defined_mean_ci(sub[measure])
        summary_rows.append({
            "mechanism": mechanism,
            "measure": measure,
            **stats,
        })

summary = pd.DataFrame(summary_rows)
SUMMARY_PATH = OUT / "v0231_nan_safe_mechanism_component_summary.csv"
summary.to_csv(SUMMARY_PATH, index=False)

core_summary = summary[summary["measure"].isin(CORE_MEASURES)].copy()
display(core_summary.round(6))

# Hard requirement of this repair: any previously NaN mechanism-level CI with
# at least one defined query must now be finite.
for _, r in summary.iterrows():
    if r["n_defined_queries"] > 0:
        assert np.isfinite(r["mean"])
        assert np.isfinite(r["ci95_low"])
        assert np.isfinite(r["ci95_high"])

print("MECHANISM SUMMARY RECOMPUTE — PASS")

In [ ]:
# Cell 9 — Recompute round profiles with explicit defined coverage
ROUND_MEASURES = [
    "state_direct_norm",
    "state_prop_norm",
    "state_component_cosine",
    "state_cancellation_ratio",
    "candidate_direct_jaccard",
    "candidate_prop_jaccard",
    "feedback_direct_cosdist",
    "feedback_prop_cosdist",
    "utility_direct_abs",
    "utility_prop_abs",
]

round_rows = []

for mechanism in MECHANISMS:
    for t in range(4):
        sub = query_mech_round[
            query_mech_round["mechanism"].eq(mechanism)
            & query_mech_round["round"].eq(t)
        ]
        for measure in ROUND_MEASURES:
            stats = bootstrap_defined_mean_ci(sub[measure])
            round_rows.append({
                "mechanism": mechanism,
                "round": int(t),
                "measure": measure,
                **stats,
            })

round_summary = pd.DataFrame(round_rows)
ROUND_PATH = OUT / "v0231_nan_safe_round_profiles.csv"
round_summary.to_csv(ROUND_PATH, index=False)

display(
    round_summary[
        round_summary["measure"].isin([
            "state_direct_norm",
            "state_prop_norm",
            "state_component_cosine",
            "state_cancellation_ratio",
        ])
    ].round(6)
)

print("ROUND PROFILE RECOMPUTE — PASS")

In [ ]:
# Cell 10 — NaN-safe paired query-level intervention differences
PAIRS = [
    ("e5_representation", "e5_nprobe"),
    ("e5_representation", "e5_hnsw"),
    ("e5_nprobe", "e5_hnsw"),
]

wide = query_mech.pivot(index="query_id", columns="mechanism", values=CORE_MEASURES)
rng_pair = np.random.default_rng(SEED + 2310)

pair_rows = []

for A, B in PAIRS:
    for measure in CORE_MEASURES:
        xa = wide[(measure, A)]
        xb = wide[(measure, B)]
        mask = xa.notna() & xb.notna()
        d = (xa[mask] - xb[mask]).to_numpy(np.float64)

        n_total = int(len(xa))
        n_paired = int(len(d))

        if n_paired == 0:
            point = lo = hi = np.nan
        else:
            point = float(d.mean())
            boots = np.empty(BOOTSTRAP_REPS, dtype=np.float64)
            for b in range(BOOTSTRAP_REPS):
                idx = rng_pair.integers(0, n_paired, size=n_paired)
                boots[b] = float(d[idx].mean())
            lo, hi = np.quantile(boots, [0.025, 0.975])

        pair_rows.append({
            "A": A,
            "B": B,
            "measure": measure,
            "mean_A_minus_B": point,
            "ci95_low": float(lo) if np.isfinite(lo) else np.nan,
            "ci95_high": float(hi) if np.isfinite(hi) else np.nan,
            "n_total_queries": n_total,
            "n_paired_defined": n_paired,
            "paired_coverage": n_paired / n_total if n_total else np.nan,
        })

pairwise = pd.DataFrame(pair_rows)
PAIR_PATH = OUT / "v0231_nan_safe_pairwise_query_differences.csv"
pairwise.to_csv(PAIR_PATH, index=False)

display(pairwise.round(6))

In [ ]:
# Cell 11 — Mechanism evidence audit, without inventing a theorem
rep_vs_search = pairwise[
    pairwise["A"].eq("e5_representation")
    & pairwise["B"].isin(["e5_nprobe", "e5_hnsw"])
].copy()

rep_vs_search["ci_excludes_zero"] = (
    (rep_vs_search["ci95_low"] > 0)
    | (rep_vs_search["ci95_high"] < 0)
)

# Direct-perturbation measures expected to be interpreted as magnitude/profile evidence,
# not as a causal proof of H3 sign.
direct_measures = [
    "state_direct_norm",
    "candidate_direct_jaccard",
    "feedback_direct_cosdist",
    "utility_direct_abs",
]

direct_view = rep_vs_search[
    rep_vs_search["measure"].isin(direct_measures)
].copy()

display(direct_view.round(6))

audit = {
    "status": "ARC_V0231_ANALYSIS_REPAIR_COMPLETE",
    "parent_v023_status": json.loads(V023_REPORT.read_text()).get("status"),
    "n_queries": 3316,
    "retrieval_rerun": False,
    "undefined_values_imputed": False,
    "core_rep_vs_search_effort_comparisons": int(len(rep_vs_search)),
    "core_nonzero_CI_count": int(rep_vs_search["ci_excludes_zero"].sum()),
    "direct_profile_comparisons": int(len(direct_view)),
    "direct_profile_nonzero_CI_count": int(direct_view["ci_excludes_zero"].sum()),
    "interpretation": (
        "Use persistent/larger common-state direct perturbation as operator-level empirical "
        "support only if the repaired estimates and round profiles remain coherent. "
        "Do not claim that lower cancellation causes representation expansion; the observed "
        "cancellation/alignment pattern does not support that simple explanation."
    ),
    "test_accessed": False,
    "test_relevance_accessed": False,
}

print(json.dumps(audit, indent=2))

In [ ]:
# Cell 12 — Generate compact manuscript-facing tables
# Table A: core mechanism-level values
table_a = (
    core_summary[
        core_summary["measure"].isin([
            "state_direct_norm",
            "state_prop_norm",
            "state_component_cosine",
            "state_cancellation_ratio",
            "candidate_direct_jaccard",
            "feedback_direct_cosdist",
            "utility_direct_abs",
        ])
    ]
    .copy()
)

table_a["estimate_ci"] = table_a.apply(
    lambda r: (
        "undefined"
        if not np.isfinite(r["mean"])
        else f'{r["mean"]:.6f} [{r["ci95_low"]:.6f}, {r["ci95_high"]:.6f}]'
    ),
    axis=1,
)

TABLE_A_PATH = OUT / "v0231_manuscript_core_component_table.csv"
table_a.to_csv(TABLE_A_PATH, index=False)

# Table B: direct-state round profile only; concise enough for paper/appendix.
table_b = round_summary[
    round_summary["measure"].isin(["state_direct_norm", "state_prop_norm"])
].copy()
TABLE_B_PATH = OUT / "v0231_manuscript_round_profile_table.csv"
table_b.to_csv(TABLE_B_PATH, index=False)

display(table_a[[
    "mechanism","measure","estimate_ci",
    "n_defined_queries","n_total_queries","undefined_fraction"
]])

## Manuscript integration guidance

If the repaired results preserve the v0.23 pattern, the evidence supports the following **bounded** interpretation:

> Common-state replay operationalizes Eq. 6 and shows that representation approximation produces substantially larger direct fidelity perturbations than either search-effort intervention across state, candidate, feedback, and utility spaces. These direct perturbations remain comparatively persistent over later rounds, whereas the nprobe and HNSW direct-state perturbations drop sharply after the initial round while propagated state differences remain. The result provides operator-level empirical support for the mechanism boundary, but it is not a contraction theorem and does not by itself establish that direct perturbation magnitude causally determines the sign of H3abs.

Do **not** claim:

- that representation expansion is caused by weaker cancellation;
- that component alignment is universally stronger for representation approximation;
- that the common-state audit was part of the original preregistered confirmation;
- that the result is a universal law across ANN methods or feedback operators.

For `state_component_cosine`, round 0 is structurally undefined when the propagated component is zero. Report later-round profiles or aggregate-over-defined-rounds coverage, rather than assigning a numerical zero.

In [ ]:
# Cell 13 — Final report / hashes
REPORT = {
    **audit,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "protocol_sha256": PROTOCOL_SHA,
    "parent_artifacts": {
        "replay_path": str(REPLAY_PATH),
        "replay_sha256": sha256_file(REPLAY_PATH),
        "v023_report_sha256": sha256_file(V023_REPORT),
        "v023_protocol_sha256": sha256_file(V023_PROTOCOL),
    },
    "outputs": {
        "event_undefined_diagnostics": str(EVENT_UNDEF_PATH),
        "query_level_coverage": str(QUERY_COVERAGE_PATH),
        "mechanism_summary": str(SUMMARY_PATH),
        "round_profiles": str(ROUND_PATH),
        "pairwise_query_differences": str(PAIR_PATH),
        "manuscript_core_component_table": str(TABLE_A_PATH),
        "manuscript_round_profile_table": str(TABLE_B_PATH),
    },
}

REPORT_PATH = OUT / "v0231_final_report.json"
REPORT_PATH.write_text(json.dumps(REPORT, indent=2))

artifact_paths = [
    PROTOCOL_PATH,
    EVENT_UNDEF_PATH,
    QUERY_COVERAGE_PATH,
    SUMMARY_PATH,
    ROUND_PATH,
    PAIR_PATH,
    TABLE_A_PATH,
    TABLE_B_PATH,
    REPORT_PATH,
]

hash_rows = []
for p in artifact_paths:
    hash_rows.append({
        "file": p.name,
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

hash_df = pd.DataFrame(hash_rows)
HASH_PATH = OUT / "V0231_ARTIFACT_SHA256.csv"
hash_df.to_csv(HASH_PATH, index=False)

print("=" * 88)
print("ARC-v0.23.1 NaN-safe statistical recompute — COMPLETE")
print("output:", OUT)
print("retrieval rerun:", False)
print("test accessed:", False)
print("=" * 88)
display(hash_df)

## What to send back

After execution, send the executed notebook or these files:

- `v0231_nan_safe_mechanism_component_summary.csv`
- `v0231_nan_safe_round_profiles.csv`
- `v0231_nan_safe_pairwise_query_differences.csv`
- `v0231_query_level_coverage.csv`
- `v0231_final_report.json`

This should take minutes, not hours, because it performs **analysis only** on the existing ~89 MB replay parquet and does not reload FAISS indexes.